# 03 - Capacity and Resilience Analysis

CRISP-DM stage covered: Modeling and stress testing capacity assumptions.

In [1]:
import sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loading import load_clean_hospital_data

In [2]:
df = load_clean_hospital_data()

cap = (
    df.groupby('STATE')
    .agg(
        facilities=('STATE', 'size'),
        total_beds=('TOTAL_BEDS', 'sum'),
        burn_beds=('BURN_BEDS', 'sum')
    )
)

cap['avg_total_beds_per_facility'] = cap['total_beds'] / cap['facilities']
cap = cap.sort_values('total_beds', ascending=False)
cap.head(15)

,facilities,total_beds,burn_beds,avg_total_beds_per_facility
STATE,,,,
TX,48,22794,206.0,474.875000
CA,62,22552,140.0,363.741935
FL,36,19224,85.0,534.000000
NY,36,18413,143.0,511.472222
IL,57,18258,77.0,320.315789
MI,36,16872,69.0,468.666667
PA,36,15191,83.0,421.972222
OH,26,14406,94.0,554.076923
MO,20,9605,93.0,480.250000


In [3]:
# Resilience proxy: simulate loss of largest-bed facility in each state.
state_top = (
    df[['STATE', 'TOTAL_BEDS']]
    .dropna(subset=['TOTAL_BEDS'])
    .groupby('STATE')['TOTAL_BEDS']
    .max()
    .rename('largest_single_facility_beds')
)

resilience = cap.join(state_top, how='left')
resilience['post_loss_capacity'] = resilience['total_beds'] - resilience['largest_single_facility_beds']
resilience['capacity_loss_pct'] = (
    resilience['largest_single_facility_beds'] / resilience['total_beds'] * 100
).replace([np.inf, -np.inf], np.nan)

resilience.sort_values('capacity_loss_pct', ascending=False).head(20)

,facilities,total_beds,burn_beds,avg_total_beds_per_facility,largest_single_facility_beds,post_loss_capacity,capacity_loss_pct
STATE,,,,,,,
VT,1,620,9.0,620.000000,620,0,100.000000
RI,1,719,0.0,719.000000,719,0,100.000000
NM,1,556,10.0,556.000000,556,0,100.000000
DE,2,1277,0.0,638.500000,1140,137,89.271731
AK,2,574,0.0,287.000000,401,173,69.860627
ME,2,1048,6.0,524.000000,637,411,60.782443
OK,3,1264,33.0,421.333333,749,515,59.256329
HI,3,1246,7.0,415.333333,637,609,51.123596
DC,4,1920,41.0,480.000000,926,994,48.229167
